In [5]:
import json
from ACL_cycle import run_llm_batfish_gns3_cycle

from Helpers.Parse import parse_config_to_json
from Helpers.Read_Files import read_all_json_files_in_folder_Eval



In [6]:
#################### Device information to access GNS3 server: consol IP and port ####################
######################################################################################################
Consol = [ # this info extracted from GNS3 consol (they are based on the VM server)
    {
        "D_Name" : "PC1",
        "D_IP"   : "10.133.14.34",
        "D_Port" : "5052"
    },
    {
        "D_Name" : "PC2",
        "D_IP"   : "10.133.14.34",
        "D_Port" : "5054"
    },
    {
        "D_Name" : "PC3",
        "D_IP"   : "10.133.14.34",
        "D_Port" : "5056"
    },
    {
        "D_Name" : "PC4",
        "D_IP"   : "10.133.14.34",
        "D_Port" : "5058"
    },
    {
        "D_Name" : "PC5",
        "D_IP"   : "10.133.14.34",
        "D_Port" : "5060"
    },
    {
        "D_Name" : "PC6",
        "D_IP"   : "10.133.14.34",
        "D_Port" : "5062"
    },
    {
        "D_Name" : "R1",
        "D_IP"   : "10.133.14.34",
        "D_Port" : "5046"
    },
    {
        "D_Name" : "R2",
        "D_IP"   : "10.133.14.34",
        "D_Port" : "5047"
    },
    {
        "D_Name" : "R3",
        "D_IP"   : "10.133.14.34",
        "D_Port" : "5048"
    }
]

for d in Consol:
    if d["D_Name"].startswith("PC"):
        d["Type"] = "VPCS"
    else:
        d["Type"] = "Router"

Sub_Net = [ # this info extracted from GNS3 topology
    {
        "D_Name" : "PC1",
        "D_IP"   : "172.16.10.2",
        "D_Net" : "172.16.10.0/24"
    },
    {
        "D_Name" : "PC2",
        "D_IP"   : "172.16.10.3",
        "D_Net" : "172.16.10.0/24"
    },
    {
        "D_Name" : "PC3",
        "D_IP"   : "172.16.30.2",
        "D_Net" : "172.16.30.0/24"
    },
    {
        "D_Name" : "PC4",
        "D_IP"   : "172.16.30.3",
        "D_Net" : "172.16.30.0/24"
    },
    {
        "D_Name" : "PC5",
        "D_IP"   : "172.16.50.2",
        "D_Net" : "172.16.50.0/24"
    },
    {
        "D_Name" : "PC6",
        "D_IP"   : "172.16.50.3",
        "D_Net" : "172.16.50.0/24"
    }
]

In [7]:
############# Set input variables + default variables to the system: intent, etc ##############
################################################################################################

parse_config_to_json('configs/R1')#.cfg')
parse_config_to_json('configs/R2')#.cfg')
parse_config_to_json('configs/R3')#.cfg')

new_intent = input("Hi!\n Type your intent here : ").strip().lower()    
file_path           = "./configs/" #"./configs"  # Replace with your folder path
file_contents       = read_all_json_files_in_folder_Eval(file_path)
network_topology            = "./configs/TopologyFile.json"

######## Load dataset for eval #######
with open(network_topology, "r", encoding="utf-8") as f:
    network_topology_json = json.load(f) # for eval
############# Initial values #############
topology_file       = None
hostname            = None
Whole_configuration = None
extraction_result   = None
L_Name              = None
Rules               = None
direction           = None
Intf_Name           = None
user_action         = None
List_Found          = None

context_variables = {
              "new_intent" : new_intent,
              "file_path"  : file_path,
           "topology_file" : file_contents,
        "network_topology" : network_topology    
    #   "network_topology" : network_topology_json,
    #   "Eval_GT_File"     : Eval_GT
}


Hi!
 Type your intent here :  Block ICMP traffic from subnet 172.16.30.0/24 to subnet 172.16.10.0/24


In [8]:
context_variables = {
    "new_intent": new_intent, #Block ICMP traffic from subnet 172.16.30.0/24 to subnet 172.16.10.0/24
    "file_path": file_path,
    "topology_file": file_contents,
    "network_topology": network_topology_json,   
    "Sub_Net": Sub_Net,
    "devices": Consol,
    "snapshot_name": "configs",
    "enable_conflict_detection": True,
}

result = run_llm_batfish_gns3_cycle(
    context_variables=context_variables,
    batfish_host="10.133.14.31",   # or your Batfish server IP
    devices=Consol,
    max_gns3_repairs=3,
    max_batfish_iters=3,
    include_save=True,
    cleanup_candidate_on_success=False,
    verbose=True,
)

    
print(result["status"])


======== INITIAL LLM GENERATION ========
new_intent = 'block icmp traffic from subnet 172.16.30.0/24 to subnet 172.16.10.0/24'


INFO:httpx:HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"


=== DEVICE DEBUG ===
source_ip: None
src_subnet: None
dst_ip: None
dst_subnet: 172.16.30.0/24
wan_edge_router: None
router_networks: {'r1': ['172.16.10.0/24', '172.16.20.0/24'], 'r2': ['172.16.30.0/24', '172.16.20.0/24', '172.16.40.0/24'], 'r3': ['172.16.50.0/24', '172.16.40.0/24']}
[GPT:gpt-4o] Processing: 'Topology (JSON):\n        {\n  "objects": {\n    "LAN10": "172.16.10.0/24",\n    "PC1": "172.16.10.2/32",\n    "PC2": "172.16' ...


INFO:httpx:HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"


[GPT RAW OUTPUT]: 'f1/0'
[GPT:gpt-4o] Processing: 'Intent:\n        block icmp traffic from subnet 172.16.30.0/24 to subnet 172.16.10.0/24\n        \n        Chosen device:\n ' ...


...

hostname used for Batfish: R2_2
cfg_path: ./configs/R2_2.cfg
ACL searched in Batfish: ACL_FASTETHERNET1_0_IN
Interface properties DF:
               Interface    Incoming_Filter_Name Outgoing_Filter_Name
0  r2_2[FastEthernet0/0]                    None                 None
1  r2_2[FastEthernet0/1]                    None                 None
2  r2_2[FastEthernet1/0]  ACL_FASTETHERNET1_0_IN                 None
3        r2_2[Serial0/0]                    None                 None
4        r2_2[Serial0/1]                    None                 None
5        r2_2[Serial0/2]                    None                 None

=== Q0 DEBUG ===
ACL name from plan: 'ACL_FASTETHERNET1_0_IN'
Expected acceptable: [('fastethernet1/0', 'out'), ('fastethernet0/0', 'in')]
Raw attachments from Batfish: [{'interface': Interface(hostname='r2_2', interface='FastEthernet1/0'), 'direction': 'in'}]
Q0 acceptable_set: {('fastethernet0/0', 'in'), ('fastethernet1/0', 'out')}
Q0 actual_raw: [{'raw': {'interface': I

INFO:pybatfish.client.workhelper:status: WorkStatusCode.ASSIGNED
INFO:pybatfish.client.workhelper:.... no task information
INFO:pybatfish.client.workhelper:status: WorkStatusCode.ASSIGNED
INFO:pybatfish.client.workhelper:.... 2026-05-06 15:56:52.201000+02:00 Serializing 5 vendor-independent configuration structures for snapshot 66f60dbc-c976-4f43-bedc-a9ace2c6b8d4 3 / 5.
INFO:pybatfish.client.workhelper:status: WorkStatusCode.TERMINATEDNORMALLY
INFO:pybatfish.client.workhelper:.... 2026-05-06 15:56:52.201000+02:00 Deserializing objects of type 'org.batfish.datamodel.Configuration' from files 5 / 5.
INFO:pybatfish.client.session:Default snapshot is now set to configs
INFO:pybatfish.client.workhelper:status: WorkStatusCode.ASSIGNED
INFO:pybatfish.client.workhelper:.... no task information
INFO:pybatfish.client.workhelper:status: WorkStatusCode.TERMINATEDNORMALLY
INFO:pybatfish.client.workhelper:.... 2026-05-06 15:56:52.467000+02:00 Parse environment BGP tables.
INFO:pybatfish.client.work

hostname used for Batfish: R2_2
cfg_path: ./configs/R2_2.cfg
ACL searched in Batfish: ACL_FASTETHERNET1_0_IN
Interface properties DF:
               Interface Incoming_Filter_Name    Outgoing_Filter_Name
0  r2_2[FastEthernet0/0]                 None                    None
1  r2_2[FastEthernet0/1]                 None                    None
2  r2_2[FastEthernet1/0]                 None  ACL_FASTETHERNET1_0_IN
3        r2_2[Serial0/0]                 None                    None
4        r2_2[Serial0/1]                 None                    None
5        r2_2[Serial0/2]                 None                    None

=== Q0 DEBUG ===
ACL name from plan: 'ACL_FASTETHERNET1_0_IN'
Expected acceptable: [('fastethernet1/0', 'out'), ('fastethernet0/0', 'in')]
Raw attachments from Batfish: [{'interface': Interface(hostname='r2_2', interface='FastEthernet1/0'), 'direction': 'out'}]
Q0 acceptable_set: {('fastethernet0/0', 'in'), ('fastethernet1/0', 'out')}
Q0 actual_raw: [{'raw': {'interface': 

INFO:pybatfish.client.workhelper:status: WorkStatusCode.TERMINATEDNORMALLY
INFO:pybatfish.client.workhelper:.... 2026-05-06 15:56:52.583000+02:00 Begin job.
INFO:pybatfish.client.workhelper:status: WorkStatusCode.ASSIGNED
INFO:pybatfish.client.workhelper:.... no task information
INFO:pybatfish.client.workhelper:status: WorkStatusCode.TERMINATEDNORMALLY
INFO:pybatfish.client.workhelper:.... 2026-05-06 15:56:52.697000+02:00 Begin job.


run_result status: ran
top_status: ran
final, final_status
{'status': 'ok', 'stage': None, 'summary': 'OK', 'reasons': []} ok
reason: None
VERIFY OUT:

$ terminal length 0
terminal length 0
R2#

$ show access-lists
show access-lists
Extended IP access list ACL_F1_0_IN
    10 permit ip host 172.16.30.3 any
    20 deny ip host 172.16.30.3 any
Extended IP access list ACL_FASTETHERNET1_0_IN
    10 deny icmp any 172.16.30.0 0.0.0.255
Extended IP access list ACL_S0_1_OUT
    10 permit icmp host 172.16.10.3 host 172.16.30.3
R2#

$ show running-config | include access-group
show running-config | include access-group
 ip access-group ACL_S0_1_IN in
 ip access-group ACL_S0_1_OUT out
 ip access-group ACL_FASTETHERNET1_0_IN in
 ip access-group ACL_FASTETHERNET1_0_IN out
R2#
PRECHECK OUT:

$ terminal length 0
terminal length 0
R2#

$ show ip interface brief
show ip interface brief
Interface                  IP-Address      OK? Method Status                Protocol
FastEthernet0/0            unassig